# 03 - Salida estructurada

Muchas aplicaciones no necesitan texto bonito: necesitan datos confiables. Aquí convertimos respuestas del modelo en JSON para que Python pueda procesarlas.


## Paso 0: helpers

Usaremos `json.loads` para validar si la respuesta realmente es JSON.


In [ ]:
import json
import ollama

MODEL = "llama3.2"

def generar(prompt, formato=None):
    argumentos = {"model": MODEL, "prompt": prompt}
    if formato:
        argumentos["format"] = formato
    return ollama.generate(**argumentos)["response"].strip()

def mostrar_json(texto):
    datos = json.loads(texto)
    print(json.dumps(datos, indent=2, ensure_ascii=False))
    return datos


## Paso 1: respuesta libre

Si no pedimos un formato, el modelo puede responder con prosa, listas, explicaciones o una mezcla.


In [ ]:
texto = generar("Dame datos de Python: año de creación, creador y paradigma principal.")
print(texto[:500])


## Paso 2: pedir JSON en el prompt

Esto suele funcionar, pero no es una garantía absoluta. Por eso conviene validar con `json.loads`.


In [ ]:
texto = generar(
    """Dame información del lenguaje Python como un objeto JSON con estas claves:
- "nombre": nombre del lenguaje
- "anio_creacion": año de creación como número
- "creador": nombre del creador
- "paradigma": paradigma principal
Responde SOLO con JSON válido, sin texto adicional."""
)

try:
    datos = mostrar_json(texto)
except json.JSONDecodeError:
    print("No fue JSON válido:")
    print(texto)


## Paso 3: forzar JSON con Ollama

`format="json"` le indica a Ollama que la salida debe ser JSON válido.


In [ ]:
texto = generar(
    """Extrae información del siguiente texto y responde en JSON:
"JavaScript fue creado por Brendan Eich en 1995. Es multiparadigma y es el lenguaje de la web."

Usa estas claves: "nombre", "anio_creacion", "creador", "paradigma".""",
    formato="json",
)

datos = mostrar_json(texto)


## Paso 4: caso práctico, clasificar reseñas

La misma técnica sirve para tareas donde el resultado alimenta otro sistema: reportes, dashboards, filtros o bases de datos.


In [ ]:
reseñas = [
    "La pizza estaba buenísima, volvería mil veces",
    "Tardó una hora y llegó fría, pésimo servicio",
    "Normal, nada del otro mundo",
]

for reseña in reseñas:
    texto = generar(
        f"""Clasifica el sentimiento de esta reseña como "positivo", "negativo" o "neutro".
Responde en JSON con claves "sentimiento" y "confianza" de 0 a 1.

Reseña: "{reseña}" """,
        formato="json",
    )
    print(reseña)
    mostrar_json(texto)
    print()
